# Диагностика сохранённого прогноза

Прочитывает только результаты из указанной папки. Не обращается к БД, не обучает модель,
не меняет прогноз и его оценку. Требуются Python, pandas и numpy; пакет проекта не нужен.
Откройте рядом с основным ноутбуком. Проверьте `OUTPUT_DIR` и нажмите **Run All**.
Пришлите скриншоты трёх коротких блоков ниже — исходные файлы и денежные суммы не нужны.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from dataclasses import asdict, dataclass, replace
import pandas as pd


@dataclass(frozen=True)
class Config:
    db_path: str = "input/m1p_database_v2.db"
    plan_table: str | None = None  # None means raw_table
    plan_value_column: str = "target"
    plan_is_forecast: int = 0
    plan_available_column: str | None = None
    portfolio_path: str | None = None
    inflow_path: str | None = None
    operations_path: str | None = None
    calendar_path: str | None = None
    reference_path: str | None = None
    output_dir: str = "outputs/forecast"
    forecast_month: str | None = None
    development_start: str | None = None
    development_end: str | None = None
    control_start: str | None = None
    control_end: str | None = None
    fact_mode: str | None = None  # daily or cumulative, explicitly chosen by the caller
    plan_mode: str | None = None
    forecast_grain: str = "segments"  # total: forecast the complete portfolio snapshots by date
    min_train_months: int = 12
    raw_table: str = "rr1plus_data_raw"
    raw_available_column: str | None = None
    raw_date_encoding: str = "iso"  # iso, yyyymmdd, s, ms, us, ns
    segments: tuple[str, ...] = (
        "subproduct", "dpd_bucket", "pvr_bucket", "terbank", "inflow"
    )
    tolerance_pp: float = 1.0
    seed: int = 42
    reference_months: int = 3
    enable_catboost: bool = True
    enable_sarimax: bool = True
    catboost_iterations: int = 600
    catboost_depths: tuple[int, ...] = (4, 6)
    blend_weights: tuple[float, ...] = (0.0, 0.25, 0.5, 0.75, 1.0)
    country: str = "RU"
    require_all_folds: bool = True
    require_calendar_overrides: bool = False
    estimate_plan_tail: bool = True  # model features only; official unknown plans stay NULL
    allow_missing_plan_folds: bool = True  # exclude unavailable plans, report reduced validation coverage

    def resolved(self):
        if self.forecast_month is None:
            raise ValueError("Specify forecast_month, e.g. Config(forecast_month='2026-11', fact_mode='daily', plan_mode='daily')")
        target = pd.Period(self.forecast_month, "M")
        values = dict(forecast_month=str(target))
        # Six development months followed by three untouched control months.
        for name, offset in [("development_start", 9), ("development_end", 4),
                             ("control_start", 3), ("control_end", 1)]:
            value = getattr(self, name)
            values[name] = str(pd.Period(value, "M")) if value is not None else str(target - offset)
        return replace(self, **values)

    def validate(self):
        if self.forecast_grain not in {"segments", "total"}:
            raise ValueError("forecast_grain must be segments or total")
        if self.forecast_grain == "total" and any([self.portfolio_path, self.inflow_path, self.operations_path, self.reference_path]):
            raise ValueError("Total forecast currently uses raw portfolio fields and calendar_path; "
                             "remove segment-level portfolio/inflow/operations/reference paths or use segments mode")
        if self.fact_mode not in {"daily", "cumulative"} or self.plan_mode not in {"daily", "cumulative"}:
            raise ValueError("Explicitly set fact_mode and plan_mode to 'daily' or 'cumulative' after checking the source report")
        if self.plan_is_forecast not in (0, 1):
            raise ValueError("plan_is_forecast must be 0 or 1; plan rows must not be summed twice")
        if not self.segments or len(set(self.segments)) != len(self.segments):
            raise ValueError("segments must be nonempty and unique")
        if self.min_train_months < 1 or self.tolerance_pp <= 0:
            raise ValueError("Invalid training length or tolerance")
        if self.reference_months < 1 or self.catboost_iterations < 1 or not self.catboost_depths:
            raise ValueError("Reference window and model sizes must be positive")
        if any(not 1 <= d <= 10 for d in self.catboost_depths):
            raise ValueError("catboost_depths must be between 1 and 10")
        if any(not 0 <= weight <= 1 for weight in self.blend_weights):
            raise ValueError("Blend weights must be between zero and one")
        if self.raw_date_encoding not in {"iso", "yyyymmdd", "s", "ms", "us", "ns"}:
            raise ValueError("Unsupported raw_date_encoding")
        if not (pd.Period(self.development_start, 'M') <= pd.Period(self.development_end, 'M')
                < pd.Period(self.control_start, 'M') <= pd.Period(self.control_end, 'M')
                < pd.Period(self.forecast_month, 'M')):
            raise ValueError("Development, control and target periods must be ordered and disjoint")

    def as_dict(self):
        return asdict(self)


def month_start(month):
    return pd.Period(month, freq="M").start_time


def origin(month):
    return month_start(month) - pd.Timedelta(nanoseconds=1)


def days(month):
    p = pd.Period(month, freq="M")
    return pd.date_range(p.start_time, p.end_time.normalize(), freq="D")


def diagnostic_panels(result, config):
    """Small screenshot-friendly panels: percentages only, no monetary amounts."""
    c = config.resolved()
    status = result.get("plan_status") or {}
    selection = result.get("selection") or {}
    score = result.get("metrics") or {}
    def number(value):
        return "—" if value is None or pd.isna(value) else f"{float(value):.3f}"
    def table(frame, columns):
        if frame is None or frame.empty:
            return "Нет результатов."
        columns = {key: value for key, value in columns.items() if key in frame}
        return frame[list(columns)].rename(columns=columns).to_string(index=False, float_format=lambda x: f"{x:.3f}")
    metric_columns = {"month": "Месяц", "max_error_pp": "Макс.пп", "mae_pp": "MAE.пп",
                      "bias_pp": "Сдвиг.пп", "hit_rate": "Попадания", "complete": "Полный"}
    selected_id = (f"{selection['leaf']}|{selection.get('total') or 'bottom_up'}|{selection['weight']:g}"
                   if selection else None)
    lines = [f"1. ИТОГ {c.forecast_month} · screenshot-v2", f"Режим: {status.get('mode', 'неизвестен')}",
             f"Выбор: {selected_id or 'резерв'}"]
    if selection.get("weight") == 1 and selection.get("total"):
        lines.append(f"Общий итог: 100% {selection['total']}; leaf не влияет на сумму.")
    lines += [f"Оценено: {score.get('scored_days', 0)}; без оценки: {score.get('unscored_days', '—')}",
              f"Полный месяц оценён: {score.get('complete', False)}; цель на доступных датах: {score.get('available_days_goal_achieved', False)}",
              f"MAE: {number(score.get('mae_pp'))} п.п.; сдвиг: {number(score.get('bias_pp'))} п.п.; макс.: {number(score.get('max_error_pp'))} п.п.",
              f"Попадания: {number(None if score.get('hit_rate') is None else 100 * score['hit_rate'])}%; последняя дата: {score.get('last_scored_date') or '—'}; ошибка: {number(score.get('last_scored_error_pp'))} п.п."]
    comparison = result.get("comparison")
    if comparison is not None and not comparison.empty and "absolute_error_pp" in comparison:
        valid = comparison[comparison.absolute_error_pp.notna()].copy()
        errors = valid.signed_error_pp
        if len(errors) and errors.gt(0).all():
            lines.append("Прогноз выше факта на всех оценённых датах.")
        elif len(errors) and errors.lt(0).all():
            lines.append("Прогноз ниже факта на всех оценённых датах.")
        if len(valid):
            valid["period"] = np.where(pd.to_datetime(valid.date).dt.day.le(5), "1–5", "6–конец")
            groups = valid.groupby("period", sort=False).agg(days=("absolute_error_pp", "size"),
                mae_pp=("absolute_error_pp", "mean"), bias_pp=("signed_error_pp", "mean"))
            lines += ["Ошибка по части месяца:", table(groups.reset_index(),
                       {"period": "Дни", "days": "Оценено", "mae_pp": "MAE.пп", "bias_pp": "Сдвиг.пп"})]
    panels = ["\n".join(lines)]

    ranking = result.get("ranking")
    lines = ["2. ПОДБОР НА ПРОШЛЫХ МЕСЯЦАХ", "Критерий: сначала минимальный максимум ошибки, затем MAE."]
    if ranking is not None and not ranking.empty:
        ranked = ranking.copy()
        ranked["rank"] = np.arange(1, len(ranked) + 1)
        # At weight=1 all leaf choices have the same aggregate prediction.
        # Collapse these aliases for display only; never modify the selection.
        def identity(candidate):
            leaf, total, weight = str(candidate).rsplit("|", 2)
            return f"{total}|1" if float(weight) == 1 else str(candidate)
        ranked["identity"] = ranked.candidate.map(identity)
        shown = ranked.drop_duplicates("identity").head(3)
        baselines = ranked[ranked.candidate.isin([f"{leaf}|bottom_up|0" for leaf in
            ["plan_baseline", "plan_execution", "plan_execution_curve", "plan_execution_blend"]])]
        shown = pd.concat([shown, baselines]).drop_duplicates("identity")
        shown = shown.sort_values("rank")
        shown["candidate"] = shown.identity
        lines += ["Топ-3 разных общих прогнозов + простые варианты:", table(shown,
            {"rank": "№", "candidate": "Модель", "months": "Мес.", "max_error_pp": "Макс.пп", "mae_pp": "MAE.пп"})]
    else:
        lines.append("Подбор не выполнен.")
    dev = result.get("development_scores")
    if selected_id and dev is not None and not dev.empty:
        lines += ["Выбранная комбинация по месяцам подбора (не независимая проверка):",
                  table(dev[dev.candidate.eq(selected_id)], metric_columns)]
    panels.append("\n".join(lines))

    control = result.get("control")
    lines = ["3. КОНТРОЛЬ ВЫБРАННОЙ КОМБИНАЦИИ", "Попадания в таблице — доля от 0 до 1.", table(control, metric_columns)]
    if control is not None and not control.empty and "max_error_pp" in control:
        if control.max_error_pp.dropna().gt(c.tolerance_pp).any():
            lines.append(f"На контроле есть превышения {c.tolerance_pp:g} п.п.; точность не подтверждена.")
    for phase, part in status.get("validation_coverage", {}).items():
        lines.append(f"{phase}: исключены {', '.join(part.get('excluded_months', [])) or 'нет'}")
    lines += list(status.get("reasons", [])) + list(status.get("warnings", []))
    panels.append("\n".join(lines))
    model_status = result.get("model_status")
    if model_status is not None and not model_status.empty and {"model", "status"} <= set(model_status):
        missing = model_status[model_status.status.eq("unavailable")].drop_duplicates()
        if not missing.empty:
            panels.append("4. НЕДОСТУПНЫЕ МОДЕЛИ\n" + table(missing.head(8),
                {"month": "Месяц", "model": "Модель", "reason": "Причина"}))
    if result.get("diagnostic_load_warnings"):
        panels.append("НЕПОЛНЫЙ НАБОР ФАЙЛОВ\n" + "\n".join(result["diagnostic_load_warnings"]))
    return panels

def load_saved_run_diagnostics(output_dir):
    """Read saved results only. No database, training, selection or forecast writes."""
    out = Path(output_dir)
    manifest = json.loads((out / "manifest.json").read_text(encoding="utf-8"))
    config = Config(**{**manifest["config"], "output_dir": str(out)}).resolved()
    result = {"selection": manifest.get("selection"), "diagnostic_load_warnings": []}
    for key, filename in [("plan_status", "forecast_run_status.json"), ("metrics", "metrics.json")]:
        path = out / filename
        result[key] = json.loads(path.read_text(encoding="utf-8")) if path.exists() else None
        if not path.exists():
            result["diagnostic_load_warnings"].append(f"Нет {filename}.")
    for key, filename in [("comparison", "comparison_total.csv"), ("ranking", "development_ranking.csv"),
                          ("development_scores", "development_scores.csv"), ("control", "control_scores.csv"),
                          ("model_status", "model_status.csv"), ("forecast", "forecast.csv")]:
        path = out / filename
        result[key] = pd.DataFrame()
        if path.exists():
            try:
                result[key] = pd.read_csv(path, encoding="utf-8-sig", **({"usecols": ["model"]} if key == "forecast" else {}))
            except pd.errors.EmptyDataError:
                pass
        elif key in {"comparison", "forecast"} or (result.get("plan_status") or {}).get("mode") == "selected_model":
            result["diagnostic_load_warnings"].append(f"Нет {filename}.")
    return result, config

def save_run_diagnostics(result, config):
    """One support log for the finished run; refresh after evaluating new actuals."""
    forecast = result.get("forecast")
    models = sorted(forecast.model.dropna().astype(str).unique().tolist()) if forecast is not None and "model" in forecast else []
    c = config.resolved()
    settings = {name: getattr(c, name) for name in [
        "forecast_month", "fact_mode", "plan_mode", "forecast_grain", "reference_months", "min_train_months",
        "development_start", "development_end", "control_start", "control_end", "enable_catboost",
        "enable_sarimax", "estimate_plan_tail", "allow_missing_plan_folds", "output_dir"]}
    def dump(value):
        return json.dumps(value, ensure_ascii=False, indent=2, default=str)
    summary = "\n\n".join(diagnostic_panels(result, c)) + "\n"
    lines = [summary, "ПОДРОБНОСТИ · screenshot-v2", "Настройки:", dump(settings),
             "Фактически использованные модели:", dump(models),
             "Выбранная комбинация:", dump(result.get("selection")),
             "Режим, причины, предупреждения и покрытие исторической проверки:", dump(result.get("plan_status")),
             "Метрики сравнения с фактом:", dump(result.get("metrics"))]
    comparison = result.get("comparison")
    if comparison is not None and not comparison.empty:
        eligible = comparison.absolute_error_pp.notna()
        columns = ["date", "plan_cumulative", "actual_efficiency_pct", "forecast_efficiency_pct",
                   "signed_error_pp", "absolute_error_pp"]
        worst = comparison.loc[eligible].nlargest(10, "absolute_error_pp")[columns].copy()
        worst["date"] = pd.to_datetime(worst.date).dt.strftime("%Y-%m-%d")
        lines += ["Десять худших оценённых дат:", worst.to_string(index=False, float_format=lambda v: f"{v:.6f}"),
                  "Даты без оценки:", ", ".join(pd.to_datetime(comparison.loc[~eligible, "date"]).dt.strftime("%Y-%m-%d")) or "нет"]
    else:
        lines.append("Сравнения с фактом нет: передайте фактические снимки целевого месяца в df.")
    text = "\n".join(lines) + "\n"
    out = Path(c.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    (out / "diagnostics.txt").write_text(text, encoding="utf-8")
    (out / "diagnostics_summary.txt").write_text(summary, encoding="utf-8")
    return text

In [ ]:
OUTPUT_DIR = "outputs/forecast_rr_fixed_10_10"
saved_result, saved_config = load_saved_run_diagnostics(OUTPUT_DIR)
_ = save_run_diagnostics(saved_result, saved_config)
panels = diagnostic_panels(saved_result, saved_config)
print("Краткая диагностика сохранена:", Path(OUTPUT_DIR, "diagnostics_summary.txt").resolve())


In [ ]:
print(panels[0])

In [ ]:
print(panels[1])

In [ ]:
print("\n\n".join(panels[2:]))